# Tutorial 6: SBI Surrogate Learning (`sbi_npe`)

Estimated time: 30-50 minutes

## Prerequisites
`torch` and `sbi` installed.

## Learning aims
- Primary package aim: fit/evaluate an SBI backend with the same user-facing surrogate interface
- Secondary scientific aim: understand likelihood-free neural posterior estimation from simulation pairs

## Success criteria
- you can run SBI fit/eval and compare behavior against PyMC on matched inputs


## Why this tutorial matters
Each step connects the CLI workflow to scientific reasoning so you can explain not only *what* ran, but *why* results are meaningful.


## Step 1: Ensure training dataset exists


In [ ]:
%%bash
set -euo pipefail
ROOT="$(pwd)"
if [ -d tutorials/specs ] && [ -d src ]; then
  ROOT="$(pwd)"
elif [ -d specs ] && [ -d ../src ]; then
  ROOT="$(pwd)/.."
fi
cd "$ROOT"
PYTHONPATH=src python -m metamodeler.cli.main run tutorials/specs/model.toy.grid.json


## Step 2: Fit/evaluate SBI surrogate


In [ ]:
%%bash
set -euo pipefail
ROOT="$(pwd)"
if [ -d tutorials/specs ] && [ -d src ]; then
  ROOT="$(pwd)"
elif [ -d specs ] && [ -d ../src ]; then
  ROOT="$(pwd)/.."
fi
cd "$ROOT"
PYTHONPATH=src python -m metamodeler.cli.main surrogate fit tutorials/specs/surrogate.toy.sbi_npe.json
PYTHONPATH=src python -m metamodeler.cli.main surrogate eval   tutorials/specs/surrogate.toy.sbi_npe.json   --inputs '{"a":[0.25,0.75,1.25,1.75],"b":[0.2,0.6,1.0,1.4]}'   --n 200


## Step 3: Plot SBI predictive summary (graphic)


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from metamodeler.spec import SurrogateSpec
from metamodeler.surrogates import eval_surrogate

cwd = Path.cwd()
if (cwd / "tutorials/specs").exists() and (cwd / "src").exists():
    repo_root = cwd
elif (cwd / "specs").exists() and (cwd.parent / "src").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError("Run this notebook from the repo root or tutorials/ directory.")

spec_payload = json.loads((repo_root / "tutorials/specs/surrogate.toy.sbi_npe.json").read_text())
spec = SurrogateSpec.model_validate(spec_payload)
inputs = {"a": [0.25, 0.75, 1.25, 1.75], "b": [0.2, 0.6, 1.0, 1.4]}
result = eval_surrogate(spec=spec, inputs_payload=inputs, n=300)

mean = np.asarray(result['summary']['mean'], dtype=float)
std = np.asarray(result['summary'].get('std', [0.0] * len(mean)), dtype=float)
x = np.arange(len(mean))

plt.figure(figsize=(6, 4))
plt.errorbar(x, mean, yerr=std, fmt='o-', capsize=4, color='tab:orange')
plt.title('SBI surrogate predictive mean ± std')
plt.xlabel('query point index')
plt.ylabel('predicted y')
plt.grid(True, alpha=0.3)
plt.show()


## Scientific mini-lesson
- SBI is useful when likelihood is hard to write but simulation is available.
- NPE learns an approximate posterior density from simulated data.
- Differences from PyMC are expected because approximation families and training dynamics differ.


In [ ]:
%%bash
set -euo pipefail
ROOT="$(pwd)"
if [ -d tutorials/specs ] && [ -d src ]; then
  ROOT="$(pwd)"
elif [ -d specs ] && [ -d ../src ]; then
  ROOT="$(pwd)/.."
fi
cd "$ROOT"
PYTHONPATH=src python -m pytest -q tests/test_surrogate_backends.py -k sbi_npe_backend_fit_sample_and_logprob
